# Lab 5 — Deep Learning on Trial: Can You Certify a Neural Network?  🚆🧠
### AI & Data Science with GenAI (Rail) · Deep Learning — your DL mini case and safety case

In **Lab 2** you built a **detector**: it read a metro train's compressor sensors and said
*"there is an air leak right now."* It worked — and then a single `if` statement matched it,
which was the real lesson.

In **Lab 4** you built a **forecast** and learned the discipline of not letting your model
see the future.

Today you do something different again. You are not the person building the model. **You are
the person who has to decide whether it can be trusted near a railway.**

---

**The situation.** A supplier has approached the depot with a product: an **APU Health
Monitor** built on a **deep neural network**. Their pitch, more or less word for word:

> *"Unlike your current rule-based alarms, our deep model needs **no failure examples** to
> train. It learns what a healthy Air Production Unit looks like and flags **any** abnormal
> behaviour — including fault types you have never seen before. Fleet-wide, always on."*

That is a genuinely attractive claim, and the technology behind it is real. Your job is to
**build their model yourself** on data you control, then decide — with evidence — whether you
would let it anywhere near a safety-related duty.

**The data is the same data as Lab 2.** Real APU sensor readings from a metro train
(Metro do Porto, 2020), with the two real air-leak failures marked. Using data you already
know means all your attention today goes on the *model* and the *decision*, not the domain.

> **What "safety-related" means here.** Nobody is proposing the model pulls the brake. But if
> a health monitor tells a depot "this unit is fine", and the depot believes it, the model has
> quietly become part of how the railway decides what to inspect. That is enough to require an
> argument — a **safety case** — before it goes live.

**Your deliverable is not a report. It is a Safety Case** (`Safety_Case_Lab5.docx`): a
structured argument with a claim, the evidence for it, the evidence against it, the residual
risk you are accepting, and a signed recommendation. That is the document a railway
actually produces, and today you write one.

## Step 0 — Get our tools ready and load the data
Same libraries as Lab 2, and the same CSV. Two things from scikit-learn are **new**:
**`StandardScaler`** (Step 2) and **`MLPRegressor`** — the neural network (Step 4).

The loading cell below works whether you are in **Google Colab** or on a **lab VM** — it uses
the local file if there is one and fetches the dataset over the network if there is not. You do
not need to change anything.

*MLP* stands for **Multi-Layer Perceptron**: the plain, fully-connected neural network from
your Deep Learning session. It is the "hello world" of deep learning, and it is enough to
make every point this lab needs to make.

In [ ]:
# ============================================================
# STEP 0 : import our tools and load the dataset
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# --- the machine-learning toolkit ---
from sklearn.preprocessing import StandardScaler       # NEW this lab (Step 2)
from sklearn.neural_network import MLPRegressor        # NEW this lab - the neural network
from sklearn.metrics import (recall_score, precision_score,
                             roc_auc_score, roc_curve, confusion_matrix)

# ---- where the data comes from -------------------------------------------------
# On a lab VM the CSV already sits next to this notebook. In Google Colab it does not,
# so we fetch the identical file over the network instead. Either way you get the same
# 144,437 rows -- nothing about the lab changes.
import os
CSV_NAME = "metropt3_teaching.csv"
CSV_URL   = "PASTE_THE_RAW_CSV_URL_HERE"        # set once by your instructor

if os.path.exists(CSV_NAME):
    source = CSV_NAME                            # lab VM / unzipped package
elif CSV_URL.startswith("http"):
    source = CSV_URL                             # Google Colab
else:
    raise FileNotFoundError(
        "Could not find metropt3_teaching.csv.\n"
        "In Colab: click the folder icon on the left, upload the CSV, then re-run this cell."
    )
print("Loading from:", "the local file" if source == CSV_NAME else "the course repository")

# One row = ONE MINUTE of sensor readings from a real metro train's compressor (APU).
# This is the SAME file you used in Lab 2.
df = pd.read_csv(source, parse_dates=["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)   # time series: always sort by time

# The 15 sensor columns (pressures, temperatures, currents, valve states):
SENSORS = ['TP2','TP3','H1','DV_pressure','Reservoirs','Oil_temperature',
           'Motor_current','COMP','DV_eletric','Towers','MPG','LPS',
           'Pressure_switch','Oil_level','Caudal_impulses']

print("Rows and columns:", df.shape)
print("From", df["timestamp"].min(), "to", df["timestamp"].max())
df.head()

## Step 1 — Read the record before you model it
Two questions before any modelling, both of which you have asked in earlier labs:

1. **How rare is the thing we care about?** (Lab 2)
2. **Is the time record complete?** (Lab 4)

The second one has a surprise in it this time.

In [ ]:
# ---- 1a. How rare are failures?  (you know this answer from Lab 2) ----
print("Failure minutes:", int(df["airleak_failure"].sum()),
      "=", round(100 * df["airleak_failure"].mean(), 2), "% of all minutes")

# ---- 1b. Is the record complete? ----
# In Lab 4 we checked this and the answer was "no missing days" - which was lucky.
# Check it here the same way.
expected = len(pd.date_range(df["timestamp"].min(), df["timestamp"].max(), freq="1min"))
print("\nMinutes expected on a complete 1-minute clock:", expected)
print("Minutes we actually have                     :", len(df))
print("MISSING                                      :", expected - len(df))

> **Reading it:** about **31,700 minutes are missing** — roughly 18% of the clock. In Lab 4,
> a gap like that would have been alarming. Before you call it a fault, **look at when the
> gaps happen.**

In [ ]:
# Where do the gaps fall? Find every jump of more than one minute, and look at the CLOCK TIME.
gap_minutes = df["timestamp"].diff().dt.total_seconds() / 60
gaps = df.loc[gap_minutes > 1, ["timestamp"]].copy()
gaps["gap_length_min"] = (gap_minutes[gap_minutes > 1] - 1).round(0)   # minutes actually ABSENT

print("Number of gaps:", len(gaps))
print("Total minutes missing:", int(gaps["gap_length_min"].sum()))

plt.figure(figsize=(9, 3.5))
plt.hist(gaps["timestamp"].dt.hour, bins=range(25), color="#2E6E7E", edgecolor="white")
plt.title("What time of day do the gaps start?")
plt.xlabel("hour of day"); plt.ylabel("number of gaps")
plt.xticks(range(0, 25, 2)); plt.tight_layout(); plt.show()

> **Insight — a gap is not automatically a fault.** **115 of the 187 gaps start between
> midnight and 06:00.** This is a *metro train*: overnight it sits in the depot with the
> compressor off and the telemetry quiet. The missing minutes are **the train not running**, not the recorder
> breaking.
>
> That matters twice over:
> - You should **not** "repair" these the way you repaired Lab 4's impossible zeros. There is
>   nothing to repair. Inventing readings for a parked train would be inventing data.
> - But it does mean **"15 rows back" is not "15 minutes back"** anywhere near a gap. When we
>   average over time later (Step 6), we will have to say *15 minutes*, not *15 rows* — and
>   pandas has to be told which one we mean.
>
> Write this down for your safety case: *the model is trained on running hours only, so it has
> no idea what a healthy cold start looks like.*

In [ ]:
# 🔧 Your turn (think, don't copy):
# The Insight above claims the missing minutes are "the train parked overnight".
# Do not take that on trust - it is exactly the kind of claim a safety case has to check.
# Overnight parking is a SHORT gap. So: how much of the missing time is in gaps LONGER
# than 12 hours (720 minutes)?
# Approach: filter `gaps` to gap_length_min > 720, then print how many there are and what
#   they total; also print the single longest one and when it starts.
# Then, in a comment: what fraction of the missing time is NOT overnight parking,
# and what should your safety case say about it?




## Step 2 — Why a neural network insists on scaling  ⭐ new idea
This is the first thing that is genuinely different from Lab 2.

A **Random Forest** does not care about units. It only ever asks *"is this value above or
below some cut point?"*, so a sensor measured in thousands and a sensor measured in
thousandths are treated alike.

A **neural network does care**, because it multiplies every input by a weight and adds them
up. A sensor with big numbers shouts; a sensor with small numbers whispers — regardless of
which one actually matters. Look at the spread of our 15 sensors.

In [ ]:
# How different are the sensor scales?
scales = df[SENSORS].agg(["mean", "std"]).T.round(3)
scales["std_vs_smallest"] = (scales["std"] / scales["std"][scales["std"] > 0].min()).round(1)
print(scales.sort_values("std", ascending=False).to_string())

> **Reading it:** `Oil_temperature` varies by about **6.5** units; `Pressure_switch` by about
> **0.043**. That is a **~150×** difference in loudness. Untreated, the network would spend
> almost all its effort on the temperature and effectively ignore the switch.
>
> **`StandardScaler` fixes this** by rewriting every sensor as *"how many standard deviations
> from its own average is this reading?"* After scaling, every sensor has mean 0 and standard
> deviation 1 — all 15 speak at the same volume.
>
> ⚠️ **The rule that catches people out:** the scaler is **fitted on the training data only**,
> then *applied* to the test data. Fitting it on everything would let the test period's
> averages leak into training — the same sin as Lab 4's threshold tuned on the test set,
> wearing a different hat.

## Step 3 — Split by time, then throw the failures away  ⭐ new idea
We split at **1 May 2020**, exactly as in Lab 2: train on February–April (which contains the
18 April failure), test on May (which contains the 29–30 May failure).

Then comes the part that makes this **deep learning of a different kind**. We are going to
train the network on the **normal minutes only** — and delete the failure minutes from its
training data entirely.

Why on earth would you throw away your failures? Because that is the supplier's whole pitch.
Their model never learns *"this is what a leak looks like"*. It learns *"this is what healthy
looks like"* — and then flags anything that does not fit. That is called **unsupervised
anomaly detection**, and it is the reason they can claim to catch faults nobody has labelled.

In [ ]:
# ============================================================
# STEP 3 : split by time, and keep only NORMAL minutes for training
# ============================================================
cutoff = pd.Timestamp("2020-05-01")
train = df[df["timestamp"] <  cutoff]     # Feb-Apr  (contains the 18 Apr failure)
test  = df[df["timestamp"] >= cutoff]     # May      (contains the 29-30 May failure)

# The network is only ever shown HEALTHY minutes:
train_normal = train[train["airleak_failure"] == 0]

print("train         :", len(train), "minutes,", int(train['airleak_failure'].sum()), "of them failures")
print("train_normal  :", len(train_normal), "minutes  <- this is ALL the network ever sees")
print("test          :", len(test), "minutes,", int(test['airleak_failure'].sum()), "of them failures")

# y_test is the truth we grade against. The MODEL never sees it - we only use it to score.
y_test = test["airleak_failure"].values

> **Insight — count the evidence, not the rows.** 105,471 training minutes sounds like a lot
> of data. It is not a lot of **evidence**. Those minutes come from **one train**, over
> **three months**, and the thing we want to detect happens in this whole dataset exactly
> **twice**. You will be graded on **one** held-out failure episode.
>
> Whatever score you get today, it is a measurement with a sample size of **one event**. Keep
> that number in your head all lab — it is the single most important line in your safety case,
> and no amount of clever modelling makes it bigger.

In [ ]:
# ============================================================
# Fit the scaler on TRAINING NORMAL data only, then apply it everywhere
# ============================================================
scaler = StandardScaler()
scaler.fit(train_normal[SENSORS])              # <- learns the averages and spreads. TRAIN ONLY.

X_train = scaler.transform(train_normal[SENSORS])   # healthy minutes, scaled
X_test  = scaler.transform(test[SENSORS])           # the test period, TRAINING scaler

print("X_train:", X_train.shape, " X_test:", X_test.shape)
print("\nAfter scaling, training columns have mean ~0 and std ~1:")
print("  mean of each column (first 5):", np.round(X_train.mean(axis=0)[:5], 3))
print("  std  of each column (first 5):", np.round(X_train.std(axis=0)[:5], 3))

In [ ]:
# 🔧 Your turn (think, don't copy):
# The test data was scaled with the TRAINING scaler, so its columns will NOT be exactly
# mean 0 / std 1. Check by how much they drift.
# Approach: take X_test.mean(axis=0) and X_test.std(axis=0), round them, and print them
#   next to the SENSORS names (pd.DataFrame is the tidy way).
# Then, in a comment: name the TWO sensors whose means have drifted furthest from 0. For the
# top one, test the obvious explanation ("that is the May failure") instead of assuming it -
# recompute the mean with the failure minutes dropped and see how much of the drift survives.




## Step 4 — Build the neural network  ⭐ the new idea of this lab
Here is the supplier's trick, and it is a lovely one.

We train a network to do something that sounds pointless: **copy its input to its output**.
Fifteen sensor readings in, the same fifteen readings out. But we force it through a
**bottleneck** — a middle layer with only **4** neurons:

```
   15 sensors  →  10  →  4  →  10  →  15 sensors
   (input)        (      squeeze      )   (output)
```

Four numbers cannot hold fifteen. So to do the job at all, the network has to discover the
*handful of underlying things* that actually drive a healthy compressor — pressure building,
motor running, oil warming — and rebuild the fifteen readings from those. It is forced to
learn **what healthy looks like**, in compressed form.

This shape is called an **autoencoder**. And here is the payoff:

> Show it a **healthy** minute and it rebuilds it almost perfectly.
> Show it a minute unlike anything in its training and it rebuilds it **badly**.
> **How badly it rebuilds a minute is our alarm signal.**

Nobody ever told it what a leak looks like. That is the supplier's entire claim in one
sentence — and the reason this lab is worth doing carefully.

In [ ]:
# ============================================================
# STEP 4 : train the autoencoder
# ============================================================
# hidden_layer_sizes=(10, 4, 10) is the squeeze: 15 -> 10 -> 4 -> 10 -> 15
# max_iter = how many EPOCHS (full passes over the training data) it is allowed.
# random_state fixes the random starting weights -- remember this line, it matters in Step 9.
autoencoder = MLPRegressor(hidden_layer_sizes=(10, 4, 10),
                           max_iter=200,
                           random_state=0)

# The unusual bit: X is BOTH the question and the answer. "Copy yourself."
autoencoder.fit(X_train, X_train)

print("Training finished.")
print("  epochs actually used :", autoencoder.n_iter_, "(of the 200 allowed)")
print("  final training loss  :", round(autoencoder.loss_, 5))

> **It stopped early on its own.** The network used **57** of its 200 allowed epochs: it
> stopped because the loss had essentially stopped improving. That is normal and good.
>
> **"Loss"** is just the average copying error on the training data. Watch it fall.

In [ ]:
# Every neural network keeps a record of its own training. Always look at it.
plt.figure(figsize=(9, 4))
plt.plot(autoencoder.loss_curve_, lw=2, color="#2E6E7E")
plt.title("Training loss: how badly the network copies healthy minutes, epoch by epoch")
plt.xlabel("epoch"); plt.ylabel("loss (mean squared error)")
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

print("loss at epoch 1 :", round(autoencoder.loss_curve_[0], 4))
print("loss at epoch 10:", round(autoencoder.loss_curve_[9], 4))
print("loss at the end :", round(autoencoder.loss_curve_[-1], 5))

> **Reading it:** a textbook training curve — a steep drop in the first few epochs, then a
> long flat tail. The network learned most of what it was going to learn almost immediately.
>
> ⚠️ **And here is the trap this curve sets.** It is a picture of how well the model copies
> **healthy** minutes. It says *nothing whatsoever* about whether the model can spot a leak.
> A beautiful loss curve is not evidence of a working alarm. Step 9 will show you exactly how
> far apart those two things can drift.

In [ ]:
# 🔧 Your turn (think, don't copy):
# See the bottleneck actually working - by watching it succeed and fail on two minutes.
# Approach: `recon = autoencoder.predict(X_test)` gives the network's rebuilt version of every
#   test minute, in SCALED units. Pick out two minutes by timestamp:
#     an ordinary healthy one   -> 2020-05-12 12:00
#     one from inside the leak  -> 2020-05-30 04:00
#   (get each row's position with  np.where(test["timestamp"] == pd.Timestamp("..."))[0][0] )
#   Build a DataFrame indexed by SENSORS showing, for each minute, the real value, the
#   rebuilt value, and the size of the gap between them.
# Then, in a comment: which minute does the network copy better, and by how much? Name the
# sensor it gets most wrong during the leak.




## Step 5 — Reconstruction error: turning the network into a health score  ⭐ new idea
Now we turn "how badly did it copy?" into a single number per minute.

For each minute: take the 15 differences between real and rebuilt, square them (so overs and
unders both count as error), and average. That is the **reconstruction error** — and it is
the health score the whole product rests on.

**High reconstruction error = "this minute does not look like anything I was trained on."**

Note what it does *not* mean. It does not mean "air leak". It means *"unfamiliar"*. Holding
those two apart is most of the safety argument you will write later.

In [ ]:
# ============================================================
# STEP 5 : reconstruction error = our health score
# ============================================================
def reconstruction_error(model, X):
    # Mean squared difference between each row and the model's rebuilt version of it.
    rebuilt = model.predict(X)
    return ((rebuilt - X) ** 2).mean(axis=1)

error_train = reconstruction_error(autoencoder, X_train)   # on healthy TRAINING minutes
error_test  = reconstruction_error(autoencoder, X_test)    # on May

print("Reconstruction error on the healthy training minutes:")
for p in [50, 90, 99, 99.9]:
    print(f"   {p:>5}th percentile: {np.percentile(error_train, p):.4f}")

print("\nOn the test period (May + the first two days of June):")
print("   average on NORMAL minutes :", round(error_test[y_test == 0].mean(), 3))
print("   average DURING the failure:", round(error_test[y_test == 1].mean(), 3))

> **Reading it:** during the failure the average error is about **0.52**, against about
> **0.14** on normal test-period minutes — roughly **four times higher**. The signal is real.
>
> But look again at the training percentiles: the healthy 99.9th percentile is **1.59**, well
> *above* the failure average of 0.52. In plain words: **some perfectly healthy minutes look
> weirder to this model than the failure does.** That overlap is where all the false alarms
> are going to come from, and you cannot fix it by choosing a cleverer threshold.

In [ ]:
# Plot the health score across the whole test period, with the real failure shaded.
plt.figure(figsize=(13, 4))
plt.plot(test["timestamp"], error_test, lw=0.6, color="#2E6E7E", label="reconstruction error")

fail_minutes = test.loc[test["airleak_failure"] == 1, "timestamp"]
plt.axvspan(fail_minutes.min(), fail_minutes.max(), color="crimson", alpha=0.25,
            label="real air-leak failure")

plt.title("The health score across the test period (higher = less familiar to the network)")
plt.xlabel("date"); plt.ylabel("reconstruction error")
plt.legend(); plt.tight_layout(); plt.show()

> **Insight — this is the chart to put in front of management, and the chart to be honest
> about.** The error does climb in the red band. It also spikes, hard, in several places where
> nothing was wrong at all. Any horizontal line you draw across this chart will cut through
> both. **That is the product.**

In [ ]:
# 🔧 Your turn (think, don't copy):
# Find the single WORST minute of the whole test period by reconstruction error, and check
# whether it was actually a failure.
# Approach: np.argmax(error_test) gives the position of the largest error. Use it to pull
#   the matching row out of `test` (test.iloc[...]), and print its timestamp, its
#   airleak_failure value, and its error.
# Then, in a comment: was the model's single loudest alarm of the test period a real failure?
# What does your answer do to the sentence "high error means the unit is faulty"?




## Step 6 — Smooth it, then set the threshold *without* looking at the answers
Two decisions turn a score into an alarm, and both are places people cheat by accident.

**Decision 1 — smooth it.** No depot wants an alarm because one minute looked odd. We
average the score over a **15-minute window**. And because of Step 1's gaps, we must say
*15 minutes*, not *15 rows* — pandas does this properly if we give it a time index.

**Decision 2 — pick the alarm level.** This is Lab 4's lesson wearing overalls. If you try
thresholds on the May data and keep the one that scores best, **you have tuned on the test
set** and your reported score is fiction. So we take the threshold from the **training**
error distribution alone: *"alarm when the score exceeds the 99th percentile of what healthy
looked like in training."* No test data touches that choice.

In [ ]:
# ============================================================
# STEP 6 : smooth over 15 MINUTES (not 15 rows), then set the alarm level
# ============================================================
# A time index lets pandas honour the clock across the gaps we found in Step 1.
smooth_train = (pd.Series(error_train, index=pd.DatetimeIndex(train_normal["timestamp"]))
                  .rolling("15min").mean().values)
smooth_test  = (pd.Series(error_test,  index=pd.DatetimeIndex(test["timestamp"]))
                  .rolling("15min").mean().values)

# The alarm level comes from the TRAINING distribution only. The test set is not consulted.
THRESHOLD = np.percentile(smooth_train, 99)
print("Alarm threshold (99th percentile of the smoothed TRAINING score):", round(THRESHOLD, 4))

alarm = (smooth_test >= THRESHOLD).astype(int)      # 1 = the monitor is shouting
print("Minutes in alarm during the test period:", int(alarm.sum()), "of", len(alarm))

In [ ]:
# 🔧 Your turn (think, don't copy):
# Show that smoothing was worth doing, instead of assuming it.
# Approach: build a second alarm from the RAW (unsmoothed) error, choosing its level by the
#   same rule we used for the smoothed one - the 99th percentile of the TRAINING errors.
#   Then score both alarms against y_test with recall_score and precision_score.
# Then, in a comment: which is better, and by how much? Be specific - quote both numbers.




## Step 7 — Score it properly, then say it in depot language
Three numbers, and then the translation that actually matters.

- **Recall** — of the real failure minutes, how many did we catch? (Misses are the safety risk.)
- **Precision** — of the minutes we shouted about, how many were real? (Low precision burns
  the depot's time and, worse, its patience.)
- **ROC-AUC** — across *every possible* threshold, how well does the score separate failure
  minutes from normal ones? 1.0 is perfect, 0.5 is a coin toss. It is the fairest single
  summary of the model itself, because it does not depend on where you drew the line.

In [ ]:
# ============================================================
# STEP 7 : score the monitor
# ============================================================
print("At our threshold (chosen from training data only):")
print("   recall    :", round(recall_score(y_test, alarm, zero_division=0), 3))
print("   precision :", round(precision_score(y_test, alarm, zero_division=0), 3))
print()
print("Across all thresholds:")
print("   ROC-AUC   :", round(roc_auc_score(y_test, smooth_test), 3))
print()
false_alarms = int(((alarm == 1) & (y_test == 0)).sum())
missed       = int(((alarm == 0) & (y_test == 1)).sum())
print("Confusion matrix [rows = true 0/1, cols = predicted 0/1]:")
print(confusion_matrix(y_test, alarm))
print(f"\n-> {false_alarms} false-alarm minutes, {missed} missed failure minutes")
print(f"-> for every 1 real failure minute caught, {false_alarms / max(1, (alarm.sum()-false_alarms)):.0f} false ones")

In [ ]:
# The ROC curve - the picture behind the AUC number.
fpr, tpr, _ = roc_curve(y_test, smooth_test)
plt.figure(figsize=(5.5, 5))
plt.plot(fpr, tpr, lw=2, color="#2E6E7E", label=f"autoencoder (AUC = {roc_auc_score(y_test, smooth_test):.3f})")
plt.plot([0, 1], [0, 1], "k--", lw=1, label="coin toss (AUC = 0.500)")
plt.title("ROC curve: can the health score tell the two apart?")
plt.xlabel("false-alarm rate"); plt.ylabel("failures caught")
plt.legend(); plt.tight_layout(); plt.show()

> **Insight — say it the way the depot hears it.** "ROC-AUC 0.941" sounds like a triumph, and
> as a measure of the *score* it is a good result. Now translate:
>
> > *Over the test period the monitor raised about **2,400 minutes of alarm**. **224** of them were the
> > real leak. The other **2,201** were not. It also stayed quiet through **166 minutes** of a
> > genuine failure.*
>
> Roughly **ten false alarms for every real one**, and it stayed silent through **more than
> four minutes in every ten** of a genuine failure (166 of 390 — recall 0.574). Both sentences
> are true of the same model. **The first is the one that sells; the
> second is the one your safety case has to answer for.**

In [ ]:
# 🔧 Your turn (think, don't copy):
# Management asks the obvious question: "fine, then just make it more sensitive so it stops
# missing things." Find out what that actually costs.
# Approach: rebuild the alarm at the 95th percentile of smooth_train instead of the 99th,
#   then print recall, precision, and the false-alarm count for the new alarm.
# Then, in a comment: did lowering the bar fix the misses? What did it cost? Quote the
# false-alarm numbers at both levels and say whether you would accept the trade.




## Step 8 — The skeptic step: could something much dumber do this?  ⭐ the habit
You have done this in every lab and it has never once been a waste of time. In Lab 2 a single
`if` statement matched a Random Forest. So before anyone signs a purchase order, put the deep
model against two opponents that involve no learning at all.

**Opponent 1 — the control chart.** A hundred-year-old idea from factory quality control.
Work out each sensor's normal range from the training data; alarm when *any* sensor strays
more than *k* standard deviations from its own average. We already scaled the data, so this
is literally *"alarm when any scaled reading is bigger than k."* One line.

**Opponent 2 — Lab 2's sensor, on its own.** `DV_pressure`, with no model of any kind, alarmed
by the same rule as everything else in this step: above the **99th percentile of its own
smoothed training values**. Not Lab 2's hand-picked `> 0.5` — that number was chosen by a human
who already knew the answer, and using it here would make the comparison unfair in the other
direction.

In [ ]:
# ============================================================
# STEP 8 : two opponents that do no learning at all
# ============================================================
# FAIRNESS IS THE WHOLE POINT OF THIS STEP, so both opponents get exactly the treatment our
# model got: the same 15-minute smoothing, and an alarm level taken from the 99th percentile
# of their own TRAINING distribution. Compare a smoothed model against a raw rival and you
# will "prove" whatever you like -- we come back to that in the Challenge.
def smooth_train_test(train_values, test_values):
    st = pd.Series(train_values, index=pd.DatetimeIndex(train_normal["timestamp"])).rolling("15min").mean().values
    te = pd.Series(test_values,  index=pd.DatetimeIndex(test["timestamp"])).rolling("15min").mean().values
    return st, te

# --- Opponent 1: control chart. X is already in "standard deviations from normal". ---
chart_train, chart_score = smooth_train_test(np.abs(X_train).max(axis=1), np.abs(X_test).max(axis=1))

# --- Opponent 2: Lab 2's sensor, read on its own, with no model of any kind ---
dv_train, dv_score = smooth_train_test(train_normal["DV_pressure"].values, test["DV_pressure"].values)

# Same rule for all three: alarm above the 99th percentile of the training score.
chart_alarm = (chart_score >= np.percentile(chart_train, 99)).astype(int)
dv_alarm    = (dv_score    >= np.percentile(dv_train,    99)).astype(int)

# Print every alarm level, so the fairness claim above can be checked rather than trusted.
print("Alarm levels, each the 99th percentile of that detector's own smoothed TRAINING score:")
print(f"   autoencoder reconstruction error : {np.percentile(smooth_train, 99):.4f}")
print(f"   control chart (max |z|)          : {np.percentile(chart_train, 99):.3f}")
print(f"   DV_pressure                      : {np.percentile(dv_train, 99):.3f}")


print("ROC-AUC — all three scores smoothed identically, so this is a fair race:")
for name, score in [("deep autoencoder", smooth_test), ("control chart (no ML)", chart_score),
                    ("DV_pressure, on its own", dv_score)]:
    print(f"   {name:26s} {roc_auc_score(y_test, score):.3f}")

print("\nAt an alarm level chosen the same honest way for each (99th pct of TRAINING score):")
for name, a in [("deep autoencoder", alarm), ("control chart", chart_alarm), ("DV_pressure alone", dv_alarm)]:
    fa = int(((a == 1) & (y_test == 0)).sum())
    ms = int(((a == 0) & (y_test == 1)).sum())
    print(f"   {name:20s} recall={recall_score(y_test, a, zero_division=0):.3f}"
          f"  precision={precision_score(y_test, a, zero_division=0):.3f}"
          f"  false-alarm minutes={fa:5d}  missed={ms}")

> **Insight — read this slowly, because it is the finding of the lab.**
>
> | Detector (identical treatment) | ROC-AUC | recall | precision | false-alarm min | missed |
> |---|---|---|---|---|---|
> | Deep autoencoder | 0.941 | 0.574 | 0.092 | 2,201 | **166** |
> | Control chart (no ML) | 0.949 | 0.990 | 0.148 | 2,223 | 4 |
> | `DV_pressure`, on its own | **0.957** | **1.000** | **0.152** | **2,172** | **0** |
>
> Every score smoothed the same way, every alarm level chosen by the same rule from training
> data only. Judged like that, **the deep model comes last on every column.** The winner is a
> single raw sensor, read straight off the train — the one Lab 2's `if` statement used. It
> caught **every minute** of the failure; the neural network missed 166 of 390.
>
> Note what it took to see that. Earlier in the build of this very lab, the autoencoder was
> compared against *unsmoothed* rivals, and on that comparison it looked competitive. The
> model did not change. **The comparison did.**
>
> **Now look hard at what won.** `DV_pressure` is the sensor Lab 2 identified as *the* air-leak
> indicator — picked by a human who already knew which fault was in the data. A fixed level on
> one pressure sensor cannot, by construction, detect an oil leak, a seized valve or an
> electrical drift. The autoencoder at least has a *mechanism* for those. So the fair
> comparison you just ran is fair arithmetically and loaded conceptually: **the rival already
> knows the answer to the only question being asked.** Hold that thought — it is what
> Challenge Part 2 is about, and it is the strongest thing that can honestly be said for the
> supplier.
>
> None of this means autoencoders are bad engineering. It means **this evidence does not
> support buying this one**. Those are different sentences, and a good safety case makes the
> second without pretending it is the first.

In [ ]:
# 🔧 Your turn (think, don't copy):
# The supplier pushes back: "you smoothed our competitors too - that flatters them. Raw, they
# are much noisier, and raw is how a threshold alarm really runs."
# Find out whether smoothing is what is carrying the opponents.
# Approach: compute the ROC-AUC of the RAW (unsmoothed) control-chart score
#   (np.abs(X_test).max(axis=1)), the RAW DV_pressure values, and the RAW autoencoder error
#   (error_test), and put each beside its smoothed figure.
# Then, in a comment: which detector gains MOST from smoothing? What does that do to the
# supplier's objection?




## Step 9 — The seed lottery: the failure mode that is specific to deep learning  ⭐ new idea
Now the part that has no equivalent in Lab 2.

A neural network starts from **random weights**. `random_state=0` in Step 4 fixed which random
start we got. That line looked like housekeeping. It was not.

Retrain the identical architecture on the identical data, changing nothing but that number,
and see what comes out.

> **A confession first.** This notebook has used `random_state=0` throughout, and every score
> you have quoted so far came from it. We are about to show you that this choice was not
> innocent — and that we should have told you sooner. Watch what your headline number does.

In [ ]:
# ============================================================
# STEP 9 : same code, same data, five different random starts
# ============================================================
# (This trains 5 networks - it takes a few tens of seconds.)
results = []
for seed in [0, 1, 2, 3, 4]:
    net = MLPRegressor(hidden_layer_sizes=(10, 4, 10), max_iter=200, random_state=seed)
    net.fit(X_train, X_train)

    e_tr = reconstruction_error(net, X_train)
    e_te = reconstruction_error(net, X_test)
    s_tr = pd.Series(e_tr, index=pd.DatetimeIndex(train_normal["timestamp"])).rolling("15min").mean().values
    s_te = pd.Series(e_te, index=pd.DatetimeIndex(test["timestamp"])).rolling("15min").mean().values

    a = (s_te >= np.percentile(s_tr, 99)).astype(int)
    results.append({
        "seed":           seed,
        "training loss":  round(net.loss_, 5),
        "ROC-AUC":        round(roc_auc_score(y_test, s_te), 3),
        "recall":         round(recall_score(y_test, a, zero_division=0), 3),
        "precision":      round(precision_score(y_test, a, zero_division=0), 3),
        "false alarms":   int(((a == 1) & (y_test == 0)).sum()),
    })

seed_table = pd.DataFrame(results).set_index("seed")
print(seed_table.to_string())

> **Insight — three things here, and all of them belong in your safety case.**
>
> **1. The same code gives you a different product every time.** Look down the recall column
> before you read on. Seed 1 produces a monitor that catches *nothing at all* — it sat
> silently through the entire failure — and there is no error message, no warning, nothing on
> the loss curve to tell you. It just quietly does not work.
>
> **2. Training loss does not rank the models.** Find the seed with the **lowest** training
> loss — by the only number you can see *while training*, that is your best model. Now read
> across its row to the AUC and the false-alarm count. The thing you watch during training is
> not the thing you are buying.
>
> **3. So what was our "AUC 0.941"?** It was seed 0 — and seed 0 is, as it happens, tied for
> the best of the five. A different engineer, on the same Tuesday, with the same code and the
> same data, would have reported a distinctly worse number and might have reached the opposite
> conclusion. Our headline was a **draw in a lottery we never mentioned we were running.**
>
> The practical consequence: the seed, the library versions and the training data have to be
> versioned together as part of the delivered artefact, and **"we retrained it" is a change
> that needs re-approval, not a maintenance task.** If you cannot reproduce the exact model
> you tested, you have not tested the model you are running.

In [ ]:
# 🔧 Your turn (think, don't copy):
# Quantify the lottery so you can put a number, not an adjective, in your safety case.
# Approach: from seed_table, print the min, max and spread (max - min) of the ROC-AUC and
#   of the recall columns.
# Then, in a comment: write the one sentence you would put in a safety case to describe how
# much this model's performance depends on a number nobody outside the dev team ever sees.




## Step 10 — "Why did it alarm?" — and why that question is hard to answer
The depot's first question about any alarm will be *"what is wrong with the unit?"*

In Lab 2 you could answer that: the Random Forest published `feature_importances_`. A neural
network has no such thing. What we *can* do is ask **which sensors it was copying worst** when
it was upset — a rough, indirect substitute for an explanation.

In [ ]:
# ============================================================
# STEP 10 : which sensors is the network unhappy about?
# ============================================================
per_sensor = pd.DataFrame((autoencoder.predict(X_test) - X_test) ** 2, columns=SENSORS)

blame = pd.DataFrame({
    "normal minutes":  per_sensor[y_test == 0].mean().round(3),
    "failure minutes": per_sensor[y_test == 1].mean().round(3),
})
blame["times worse"] = (blame["failure minutes"] / blame["normal minutes"]).round(1)
blame = blame.sort_values("failure minutes", ascending=False)
print(blame.to_string())

plt.figure(figsize=(8, 5))
blame["failure minutes"].head(8)[::-1].plot(kind="barh", color="#2E6E7E")
plt.title("Sensors the network copies worst during the failure")
plt.xlabel("mean squared reconstruction error"); plt.tight_layout(); plt.show()

> **Reading it — and reading it carefully.** `DV_pressure` tops the list, which is reassuring:
> it is the same sensor Lab 2's one-line rule used, and the same one a fitter would check for
> an air leak. The model is looking in a sensible place.
>
> But notice the second column. `Oil_temperature` is one of the model's *largest* error
> sources during the failure (0.477) — and it is **exactly as large on normal minutes**
> (0.490), a ratio of **1.0**. The network is permanently confused by oil temperature and
> always has been. On a bar chart of "what the model is unhappy about", that confusion sits
> near the top looking like evidence.
>
> **This is not an explanation, and you must not let it be presented as one.** It tells you
> where the model's arithmetic went wrong, not what is wrong with the compressor. A fitter
> sent to investigate "high Oil_temperature error" would find nothing, twice a week, forever.

In [ ]:
# 🔧 Your turn (think, don't copy):
# Separate the sensors that genuinely CHANGE during a failure from the ones the model is
# always bad at.
# Approach: you already have the "times worse" column. Sort the blame table by it instead of
#   by raw error, and look at the top of that ordering.
# Then, in a comment: which sensor rises the most RELATIVE to its normal level? Is it the
# same sensor as the raw ranking gives? Which of the two orderings would you put in front of
# a fitter, and why?




## 🏁 Challenge (do BOTH)

**Part 1 — Put a price on it, and watch the answer move three times.**
A false alarm costs **1**. A missed failure minute costs **50**.

**(a)** Cost the autoencoder at the 90th, 95th, 97th, **98th**, 99th and 99.9th percentile
thresholds. Is the cost curve well behaved? What would you have concluded if you had used a
grid without the 98th in it?

**(b)** Then question the unit. We have been counting alarm **minutes** — but a depot does not
send a fitter out once a minute, it sends one out once per **alarm**. Recount the autoencoder's
false alarms as **call-outs** (contiguous runs of alarm) instead of minutes. How big is the
difference?

**(c)** Now the trap, and it is the reason this Challenge exists. Compare the autoencoder's
call-outs against the **raw, unsmoothed** control chart and `DV_pressure > 0.5` — then against
the properly smoothed versions from Step 8. Report both comparisons. One of them says the deep
model needs seven times fewer visits. The other does not. Say which comparison is the honest
one and what the dishonest one was actually measuring.

Then give your recommendation, and make sure it survives all three parts.

*(Approach: count alarm episodes by finding where the alarm array switches from 0 to 1 —
`np.diff(np.concatenate([[0], a, [0]])) == 1`.)*

**Part 2 — Test the claim that sold the product.**
The supplier's headline was: *"it flags any abnormal behaviour, **including fault types you
have never seen before**."* Everything you have measured today concerns **one** fault type,
seen **twice**.

Design the experiment that would actually test that claim. Say exactly what data you would
need, how you would split it, and what result would make you believe them. Then state plainly
whether the dataset in front of you can run that experiment.

*(There is no code to write for Part 2 — but there is a short, precise answer, and getting it
right is the most valuable thing in this lab.)*

In [ ]:
# 🏁 Challenge — your workspace
# Part 1 (a) the threshold grid, (b) minutes vs call-outs, (c) the trap.
# Part 2 is written, not coded - put your answer in the Safety Case document.




## ✅ Now write your Safety Case
You have built the supplier's model, measured it honestly, and found things their brochure
does not mention. The **deliverable is your Safety Case** (`Safety_Case_Lab5.docx`), written
in **your own words**.

A safety case is not a summary of what you did. It is an **argument** that someone senior can
disagree with, structured so the disagreement lands in the right place:

| Section | What goes in it |
|---|---|
| **Claim** | The single sentence you are asking someone to sign. Be precise about scope. |
| **Evidence for** | The measurements that support it — with the numbers. |
| **Evidence against** | The measurements that undermine it. This section is the point of the document. |
| **Cost of being wrong** | What an error costs — and who chose the unit that number is in. |
| **Assumptions & limitations** | What has to be true for your claim to hold. |
| **Residual risk** | What can still go wrong if you are believed, and who carries it. |
| **Recommendation** | Deploy, trial, or decline — and what would change your mind. |

Three things separate a good safety case from a bad one:

1. **A claim narrow enough to be true.** "The monitor detects APU failures" is not
   defensible on what you have. Something like *"on one unit, on one fault type, on one
   held-out episode, the monitor flagged 57% of failure minutes at roughly ten false-alarm
   minutes per real one, and was out-detected by a single unprocessed sensor"* is — and it is
   much less flattering, which is exactly the information the decision needs.
2. **Evidence against, stated as plainly as evidence for.** The seed lottery, the no-ML
   baselines out-detecting the model, the untested generalisation claim, and the fact that
   the cost figure moves by a factor of several depending on choices you made. If a reviewer
   finds a weakness you did not list, your whole document loses its credibility — including
   the parts that were right.
3. **A recommendation you would put your name to**, with the conditions attached.

> ⚠️ **Do not paste code or cell outputs into the safety case.** Numbers are your *evidence*;
> the document is your *argument*. A safety case that is pasted output earns no marks — and in
> the real world it earns no signature.

Save this notebook (File ▸ Save) and submit it **together with** your Safety Case.

*Data: MetroPT-3 (Metro do Porto APU sensors, 2020). UCI ML Repository, dataset 791.
Licence: Creative Commons Attribution 4.0 (CC BY 4.0) — free to use, with attribution.*